# Question 12 - 380. Insert Delete GetRandom O(1)

Implement the `RandomizedSet` class:

- `RandomizedSet()` Initializes the `RandomizedSet` object.
- `bool insert(int val)` Inserts an item `val` into the set if not present. Returns `true` if the item was not present, `false` otherwise.
- `bool remove(int val)` Removes an item `val` from the set if present. Returns `true` if the item was present, `false` otherwise.
- `int getRandom()` Returns a random element from the current set of elements (it's guaranteed that at least one element exists when this method is called). Each element must have the **same probability** of being returned.

You must implement the functions of the class such that each function works in **average** `O(1)` time complexity.

**Example 1:**

    Input
    ["RandomizedSet", "insert", "remove", "insert", "getRandom", "remove", "insert", "getRandom"]
    [[], [1], [2], [2], [], [1], [2], []]
    Output
    [null, true, false, true, 2, true, false, 2]

    Explanation
    RandomizedSet randomizedSet = new RandomizedSet();
    randomizedSet.insert(1); // Inserts 1 to the set. Returns true as 1 was inserted successfully.
    randomizedSet.remove(2); // Returns false as 2 does not exist in the set.
    randomizedSet.insert(2); // Inserts 2 to the set, returns true. Set now contains [1,2].
    randomizedSet.getRandom(); // getRandom() should return either 1 or 2 randomly.
    randomizedSet.remove(1); // Removes 1 from the set, returns true. Set now contains [2].
    randomizedSet.insert(2); // 2 was already in the set, so return false.
    randomizedSet.getRandom(); // Since 2 is the only number in the set, getRandom() will always return 2.

**Constraints:**

- `-2^31 <= val <= 2^31 - 1`
- At most `2 * 10^5` calls will be made to `insert`, `remove`, and `getRandom`.
- There will be **at least one** element in the data structure when `getRandom` is called.

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Design: list (for random pick) + dict value→index (for lookup), swap-with-last delete

📘 **Revise first:** [Pattern G · hash map + array](./0-concepts-array-and-string.ipynb) · [Part 3.4–3.6 · list costs](./0-concepts-array-and-string.ipynb) · [Part 6 · set/dict costs](./0-concepts-array-and-string.ipynb) in the concepts notebook

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (list only) | insert/remove O(n), getRandom O(1) | O(n) |
| 1️⃣ Brute force (set only) | insert/remove O(1), getRandom O(n) | O(n) |
| 2️⃣ Optimized (list + dict) | all O(1) average | O(n) |

> 🧠 = plain-English intuition (how you'd explain it to a friend) · 🗣️ = **interview script**: what to say out loud at each step.

### 🧠 The problem in plain English

Build a little container (a "set" of numbers, no duplicates) that supports three actions, **all instantly** (O(1) on average):
1. `insert(x)`: add x if it's not already there.
2. `remove(x)`: take x out if it's there.
3. `getRandom()`: hand back a random member, **every member equally likely**.

**Why is this tricky?** Each standard Python tool is great at some of these and bad at others:

| Tool | "Is x in here?" | Pick a random member |
|---|---|---|
| **list** (numbered boxes) | slow: must look through every box | fast: pick a random box number |
| **set / dict** (hash table) | fast: jumps straight to x | slow: has no box numbers to pick from |

**Everyday picture: a coat check.** Coats hang on a **numbered rail** (the list): to pick a random coat, roll a die for a hook number. The attendant also has a **ticket book** (the dict) saying *"Alice's coat is on hook 3"*, so finding a specific coat is instant.

When Alice collects her coat from hook 3, leaving a **gap** would ruin the dice trick (you might roll an empty hook). So the attendant **moves the coat from the last hook into hook 3**, updates that coat's ticket, and the rail stays gap-free. That move is the heart of the solution.

### Step 0 · Clarify before coding

🗣️ *"This is a design question. I need insert, remove, and getRandom, all in average O(1). Let me first list what each operation needs: insert and remove need a fast membership check; getRandom needs fast access by position."*

| Question to ask | Answer | Why it matters |
|---|---|---|
| Duplicates allowed? | **No**, it's a set. Insert of an existing value returns `false`. | |
| `getRandom` on an empty set? | **Never** happens. | No need to guard. |
| "Average O(1)"? | Hash tables and list appends are O(1) **on average / amortized**. | That's acceptable here. |
| Does order matter? | **No.** | This lets us move elements around freely. |

### Step 1 · Brute force: the "obvious" way

**Try a list alone:**
- `getRandom` → `random.choice(list)` → instant ✅
- `insert` → must first check `val in list` → scans every element → **O(n)** ❌
- `remove` → `list.remove(val)` searches **and** shifts everything after it → **O(n)** ❌

**Try a set alone:**
- `insert` / `remove` → instant ✅
- `getRandom` → a set has no positions, so you must convert it to a list first → **O(n)** ❌

🗣️ *"Each structure alone gets two of the three. A list gives me random access but slow membership and deletion; a set gives me fast membership but no random access. That strongly suggests using both together, so the question becomes how to keep them in sync in O(1)."*

In [1]:
import random


class RandomizedSetBrute:
    # list only: getRandom is O(1), but insert/remove are O(n)
    def __init__(self):
        self.items = []

    def insert(self, val: int) -> bool:
        if val in self.items:          # O(n) scan
            return False
        self.items.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.items:      # O(n) scan
            return False
        self.items.remove(val)         # O(n) search + shift
        return True

    def getRandom(self) -> int:
        return random.choice(self.items)

### Step 2 · Optimize: the "aha" moment

**Use both, each for what it's good at:**
- `vals`: a **list** of the members → `random.choice(vals)` is O(1).
- `pos`: a **dict** mapping `value → its index in vals` → "is it here?" and "where is it?" are O(1).

**Insert** is easy: append to the list, and record `pos[val] = last index`.

**Remove** is the hard part. Deleting from the middle of a list shifts everything after it, which is O(n).

> 💡 **Aha: order doesn't matter, so don't delete from the middle. Fill the hole with the last element, then remove the end.**
> 1. Find where the value lives: `i = pos[val]`.
> 2. Take the **last** element, `last = vals[-1]`, and copy it into slot `i`.
> 3. Update the ticket for the moved element: `pos[last] = i`.
> 4. Drop the end of the list (`vals.pop()`, which is O(1)) and delete `pos[val]`.

The list stays **gap-free** (so `getRandom` stays fair and O(1)), and nothing ever shifts.

🗣️ *"I'll combine a list and a hash map. The list holds the values so getRandom is just random.choice. The map stores each value's index in the list so I can check membership and find it in O(1). Insert appends and records the index. For remove, instead of deleting from the middle, which would shift elements, I overwrite the removed slot with the last element, update that element's index in the map, then pop from the end. Every operation is O(1) on average, with O(n) space."*

#### 🤔 Why this approach, and why not the others?

| Option | Verdict | Plain-English reason |
|---|---|---|
| List only | ❌ | Finding and deleting require scanning and shifting. O(n). |
| Set or dict only | ❌ | No box numbers, so a fair random pick costs O(n). |
| Linked list + dict | ❌ | Deleting a known node is O(1), but there's **no "jump to the k-th item"** for random pick. |
| Sorted structure / balanced tree | ❌ | Everything becomes O(log n), and random pick needs extra machinery. |
| Leave `None` holes instead of moving the last element | ❌ | `getRandom` would sometimes hit a hole and need to retry. As holes pile up, it's no longer O(1). |
| **list + dict, swap-with-last** | ✅ **best** | Each structure covers the other's weakness, and the swap keeps the list gap-free. |

In [2]:
import random


class RandomizedSet:
    def __init__(self):
        self.vals: list[int] = []        # gap-free list: O(1) random pick
        self.pos: dict[int, int] = {}    # value -> index in vals: O(1) lookup

    def insert(self, val: int) -> bool:
        if val in self.pos:
            return False
        self.pos[val] = len(self.vals)
        self.vals.append(val)
        return True

    def remove(self, val: int) -> bool:
        if val not in self.pos:
            return False
        i = self.pos[val]
        last = self.vals[-1]
        self.vals[i] = last          # move the last element into the hole
        self.pos[last] = i           # update its "ticket"
        self.vals.pop()              # O(1): remove from the end
        del self.pos[val]            # do this AFTER updating pos[last]
        return True

    def getRandom(self) -> int:
        return random.choice(self.vals)

### Step 3 · Toy example walkthrough (tell it like a story)

Operations: `insert(10)`, `insert(20)`, `insert(30)`, `remove(10)`, `getRandom()`

| operation | vals (the rail) | pos (the ticket book) | what happened |
|---|---|---|---|
| insert(10) | [10] | {10: 0} | appended |
| insert(20) | [10, 20] | {10: 0, 20: 1} | appended |
| insert(30) | [10, 20, 30] | {10: 0, 20: 1, 30: 2} | appended |
| remove(10), steps 1–3 | [**30**, 20, 30] | {10: 0, 20: 1, 30: **0**} | 10 was at slot 0; the last value (30) moved into slot 0 |
| remove(10), step 4 | [30, 20] | {20: 1, 30: 0} | popped the end, deleted 10's ticket |
| getRandom() | | | `random.choice([30, 20])`, each ½ |

**Narrated:** "To remove 10, my map says it's in slot 0. Rather than shifting 20 and 30 left, I copy the last value, 30, into slot 0, update 30's index to 0, then pop the end of the list and delete 10 from the map. The list is still compact, so a random pick is still fair."

**Tricky case: removing the element that is already last.** Removing 20 when `vals = [30, 20]`: `i = 1`, `last = 20`. We copy 20 onto itself and set `pos[20] = 1` (harmless), then pop and `del pos[20]`. It works **only because** we delete `pos[val]` **after** updating `pos[last]`. If we deleted first, we'd accidentally put 20 back into the map.

In [3]:
random.seed(0)

for Impl in (RandomizedSetBrute, RandomizedSet):
    rs = Impl()
    ops = ["insert", "remove", "insert", "getRandom", "remove", "insert", "getRandom"]
    args = [[1], [2], [2], [], [1], [2], []]
    out = [getattr(rs, op)(*a) for op, a in zip(ops, args)]
    assert out[:3] == [True, False, True] and out[3] in (1, 2) and out[4:] == [True, False, 2]
    print(Impl.__name__, out)

# removing the last element, removing until empty
rs = RandomizedSet()
for v in [5, 6, 7]:
    rs.insert(v)
assert rs.remove(7) and rs.remove(5) and rs.remove(6)
assert rs.vals == [] and rs.pos == {}

# fuzz test against Python's set, checking list and dict stay in sync
rs, ref = RandomizedSet(), set()
for _ in range(5000):
    v = random.randint(-5, 5)
    if random.random() < 0.5:
        assert rs.insert(v) == (v not in ref); ref.add(v)
    else:
        assert rs.remove(v) == (v in ref); ref.discard(v)
    assert set(rs.vals) == ref and all(rs.vals[i] == x for x, i in rs.pos.items())

# fairness check: each of 4 values should come up about 25% of the time
from collections import Counter
rs = RandomizedSet()
for v in range(4):
    rs.insert(v)
freq = Counter(rs.getRandom() for _ in range(40000))
print("getRandom frequencies:", dict(sorted(freq.items())))
assert all(abs(c - 10000) < 600 for c in freq.values())
print("All tests passed ✅")

RandomizedSetBrute [True, False, True, 2, True, False, 2]
RandomizedSet [True, False, True, 1, True, False, 2]
getRandom frequencies: {0: 10001, 1: 10179, 2: 10008, 3: 9812}
All tests passed ✅


### Step 4 · Complexity in plain words, and wrap up

| | insert | remove | getRandom | Memory |
|---|---|---|---|---|
| List only | O(n) | O(n) | O(1) | O(n) |
| Set only | O(1) | O(1) | O(n) | O(n) |
| **list + dict** | **O(1)** avg | **O(1)** avg | **O(1)** | **O(n)** |

**Why "average" O(1)?**
- A dict (hash table) finds keys in O(1) **on average**. In rare worst cases with many collisions it can degrade, which is why the problem says "average".
- `list.append` is **amortized** O(1): occasionally the list grows and copies itself, but averaged over many appends it's constant (concepts notebook, Part 2).
- `list.pop()` from the end and `vals[i] = x` are always O(1).

**Why O(n) memory?** Each value is stored twice: once in the list and once as a dict key. That's still proportional to n.

**Likely follow-ups:**
- *Allow duplicates* (LC 381) → `pos` maps `value → set of indices`. Swap-with-last still works; update the index sets carefully.
- *Weighted random pick* (LC 528) → prefix sums plus binary search. O(log n), no longer O(1).
- *Thread safety?* → Wrap each operation in a lock.

---

#### 🗣️ Full interview script (about 60 seconds)

*"I need insert, remove, and getRandom all in average O(1).*

*A list alone gives O(1) random access but O(n) membership checks and deletions. A hash set alone gives O(1) membership but no way to pick a random element without converting to a list. So I'll use both: a list of values for random access, and a dictionary from value to its index in the list.*

*Insert checks the dictionary, appends to the list, and records the index. getRandom is random.choice on the list. Remove is the interesting part: deleting from the middle would shift elements, so instead I take the last element, move it into the removed element's slot, update its index in the dictionary, and pop the end of the list. I update the moved element before deleting the removed key so the 'remove the last element' case works.*

*Everything is O(1) on average, with O(n) space."*